# Initialization

In [1]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
%matplotlib inline
%config InlineBackend.figure_format = 'png'
%config InlineBackend.figure_format = 'retina'

# Загрузка данных

In [14]:
# старые данные
books = pd.read_parquet("goodreads/books.parquet")
interactions = pd.read_parquet("goodreads/interactions.parquet")

In [3]:
items = pd.read_parquet("items.par")
events = pd.read_parquet("events.par")

In [10]:
print(items.shape)
items.describe()

(43312, 19)


,item_id,num_pages,average_rating,ratings_count,text_reviews_count,publication_year
count,4.331200e+04,37001.0,43312.00000,43312.0,43312.000000,35891.0
mean,8.050797e+06,337.866085,3.99872,13361.85272,637.829239,2006.881084
std,9.315783e+06,256.097045,0.31284,78911.120232,2553.203776,167.257929
min,1.000000e+00,0.0,0.00000,0.0,0.000000,13.0
25%,2.544730e+05,224.0,3.81000,328.0,29.000000,2002.0
50%,3.354695e+06,313.0,4.01000,1834.0,120.000000,2008.0
75%,1.429010e+07,400.0,4.20000,7310.0,435.000000,2012.0
max,3.652450e+07,14777.0,5.00000,4899965.0,142645.000000,20136.0


In [11]:
print(events.shape)
events.describe()

(11751086, 7)


,user_id,item_id,rating
count,1.175109e+07,1.175109e+07,1.175109e+07
mean,1.215043e+06,9.824049e+06,3.948220e+00
std,1.244405e+05,9.059209e+06,9.520989e-01
min,1.000000e+06,1.000000e+00,1.000000e+00
25%,1.107128e+06,1.705480e+05,3.000000e+00
50%,1.215328e+06,8.677937e+06,4.000000e+00
75%,1.322697e+06,1.718213e+07,5.000000e+00
max,1.430584e+06,3.642107e+07,5.000000e+00


# Разбиение с учётом хронологии

Рекомендательные системы на практике работают с учётом хронологии. Поэтому поток событий для тренировки и валидации полезно делить на то, что уже случилось, и что ещё случится. Это позволяет проводить валидацию на тех же пользователях, на которых тренировались, но на их событиях в будущем.

# === Знакомство: "холодный" старт

In [18]:
old_uid = '8f50136afeb65c55cec7b3d306c24b03'

interactions.query("user_id == @old_uid")

,user_id,book_id,started_at,read_at,is_read,rating,is_reviewed
10841363,8f50136afeb65c55cec7b3d306c24b03,253058,2012-12-29,2013-01-22,True,3,True
10841364,8f50136afeb65c55cec7b3d306c24b03,10572,2012-01-02,2013-01-27,True,5,False


In [21]:
events.query("item_id == 253058 and is_read == True and rating == 3 and is_reviewed == True")

,user_id,item_id,started_at,read_at,is_read,rating,is_reviewed
902241,1411824,253058,2014-05-06,2014-05-06,True,3,True
2370789,1141610,253058,2013-07-05,2014-04-17,True,3,True
3770520,1002928,253058,2016-04-16,2016-04-24,True,3,True
5906426,1227021,253058,2015-01-11,2015-01-17,True,3,True
6133010,1346812,253058,2015-06-12,2015-06-14,True,3,True
7043494,1329312,253058,2012-12-08,2012-12-28,True,3,True
8199826,1412609,253058,2011-09-17,2011-09-25,True,3,True
9292931,1301340,253058,2017-07-23,2017-07-30,True,3,True
10841363,1241243,253058,2012-12-29,2013-01-22,True,3,True
11013442,1267186,253058,2012-02-24,2012-03-06,True,3,True


In [27]:
# зададим точку разбиения
train_test_global_time_split_date = pd.to_datetime("2017-08-01").date()

train_test_global_time_split_idx = events["started_at"] < train_test_global_time_split_date
events_train = events[train_test_global_time_split_idx]
events_test = events[~train_test_global_time_split_idx]

# количество пользователей в train и test
users_train = events_train["user_id"].drop_duplicates()
users_test = events_test["user_id"].drop_duplicates()

# количество пользователей, которые есть и в train, и в test
common_users = set(users_train).intersection(set(users_test))

print(len(users_train), len(users_test), len(common_users))

428220 123223 120858


In [28]:
cold_users = set(users_test) - set(common_users)

print(len(cold_users)) 

2365


In [45]:
from sklearn.preprocessing import MinMaxScaler

top_pop_start_date = pd.to_datetime("2015-01-01").date()

item_popularity = events_train \
    .query("started_at >= @top_pop_start_date") \
    .groupby(["item_id"]).agg(users=("user_id", "nunique"), avg_rating=("rating", "mean")).reset_index()

# нормализация пользователей и среднего рейтинга, требуется для их приведения к одному масштабу
scaler = MinMaxScaler()
item_popularity[["users_norm", "avg_rating_norm"]] = scaler.fit_transform(
    item_popularity[["users", "avg_rating"]]
)

# вычисляем popularity_score, как скор популярности со штрафом за низкий рейтинг
item_popularity["popularity_score"] = (
    item_popularity["users_norm"] * item_popularity["avg_rating_norm"]
)



# сортируем по убыванию popularity_score
item_popularity = item_popularity.sort_values(['popularity_score'], ascending=False)

# выбираем первые 100 айтемов со средней оценкой avg_rating не меньше 4
top_k_pop_items = item_popularity.query("avg_rating >= 4").head(100)
top_k_pop_items

,item_id,users,avg_rating,users_norm,avg_rating_norm,popularity_score
32387,18007564,20207,4.321275,0.496596,0.830319,0.412333
32623,18143977,19462,4.290669,0.478287,0.822667,0.393471
2,3,15139,4.706057,0.372042,0.926514,0.344702
30695,16096824,16770,4.301014,0.412126,0.825253,0.340108
1916,15881,13043,4.632447,0.320529,0.908112,0.291076
...,...,...,...,...,...,...
24837,8490112,4792,4.080968,0.117747,0.770242,0.090694
33611,18966819,4361,4.374914,0.107154,0.843729,0.090409
378,3636,4667,4.098564,0.114675,0.774641,0.088832
32835,18293427,4674,4.092640,0.114847,0.773160,0.088795


In [46]:
# добавляем информацию о книгах
top_k_pop_items_with_info = top_k_pop_items.merge(
    items.set_index("item_id")[["author", "title", "genre_and_votes", "publication_year"]], on="item_id")

with pd.option_context('display.max_rows', 20):
    display(top_k_pop_items_with_info[["item_id", "author", "title", "publication_year", "users", "avg_rating", "popularity_score", "genre_and_votes"]])

,item_id,author,title,publication_year,users,avg_rating,popularity_score,genre_and_votes
0,18007564,Andy Weir,The Martian,2014,20207,4.321275,0.412333,"{'Science Fiction': 11966, 'Fiction': 8430}"
1,18143977,Anthony Doerr,All the Light We Cannot See,2014,19462,4.290669,0.393471,"{'Historical-Historical Fiction': 13679, 'Fict..."
2,3,"J.K. Rowling, Mary GrandPré",Harry Potter and the Sorcerer's Stone (Harry P...,1997,15139,4.706057,0.344702,"{'Fantasy': 59818, 'Fiction': 17918, 'Young Ad..."
3,16096824,Sarah J. Maas,A Court of Thorns and Roses (A Court of Thorns...,2015,16770,4.301014,0.340108,"{'Fantasy': 14326, 'Young Adult': 4662, 'Roman..."
4,15881,"J.K. Rowling, Mary GrandPré",Harry Potter and the Chamber of Secrets (Harry...,1999,13043,4.632447,0.291076,"{'Fantasy': 50130, 'Young Adult': 15202, 'Fict..."
...,...,...,...,...,...,...,...,...
95,8490112,Laini Taylor,Daughter of Smoke & Bone (Daughter of Smoke & ...,2011,4792,4.080968,0.090694,"{'Fantasy': 11681, 'Young Adult': 7110, 'Roman..."
96,18966819,Pierce Brown,"Golden Son (Red Rising, #2)",2015,4361,4.374914,0.090409,"{'Science Fiction': 2613, 'Fantasy': 1372, 'Sc..."
97,3636,Lois Lowry,"The Giver (The Giver, #1)",2006,4667,4.098564,0.088832,"{'Young Adult': 10993, 'Fiction': 9045, 'Class..."
98,18293427,Gabrielle Zevin,The Storied Life of A.J. Fikry,2014,4674,4.092640,0.088795,"{'Fiction': 3795, 'Contemporary': 1100, 'Writi..."


In [51]:
cold_users_events_with_recs = \
    events_test[events_test["user_id"].isin(cold_users)] \
    .merge(top_k_pop_items[["item_id", "avg_rating"]], on="item_id", how="left")

cold_user_items_no_avg_rating_idx = cold_users_events_with_recs["avg_rating"].isnull()
cold_user_recs = cold_users_events_with_recs[~cold_user_items_no_avg_rating_idx] \
    [["user_id", "item_id", "rating", "avg_rating"]]
cold_users_events_with_recs
    

,user_id,item_id,started_at,read_at,is_read,rating,is_reviewed,avg_rating
0,1361610,6900,2017-10-09,2017-10-13,True,4,False,NaN
1,1361610,12555,2017-09-21,2017-10-11,True,3,False,NaN
2,1361610,25899336,2017-09-12,2017-09-17,True,4,True,4.427261
3,1361610,21936809,2017-08-20,2017-08-24,True,4,True,NaN
4,1361610,6952,2017-09-18,2017-09-20,True,3,False,NaN
...,...,...,...,...,...,...,...,...
9667,1178502,252499,2017-09-30,2017-10-06,True,4,False,NaN
9668,1253160,51113,2017-09-25,2017-10-07,True,4,False,NaN
9669,1253160,16181775,2017-09-24,2017-09-25,True,3,False,NaN
9670,1253160,10210,2017-09-16,2017-09-24,True,5,False,NaN


In [59]:
cold_users_events_with_recs['avg_rating'].notna().sum()

1912

In [61]:
# посчитаем метрики рекомендаций
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

rmse = np.sqrt(
    mean_squared_error(
        cold_user_recs["rating"],
        cold_user_recs["avg_rating"]
    )
)

mae = mean_absolute_error(
    cold_user_recs["rating"],
    cold_user_recs["avg_rating"]
)

print(round(rmse, 2), round(mae, 2))

0.78 0.62


In [62]:
# посчитаем покрытие холодных пользователей рекомендациями

cold_users_hit_ratio = cold_users_events_with_recs.groupby("user_id").agg(hits=("avg_rating", lambda x: (~x.isnull()).mean()))

print(f"Доля пользователей без релевантных рекомендаций: {(cold_users_hit_ratio == 0).mean().iat[0]:.2f}")
print(f"Среднее покрытие пользователей: {cold_users_hit_ratio[cold_users_hit_ratio != 0].mean().iat[0]:.2f}")

Доля пользователей без релевантных рекомендаций: 0.59
Среднее покрытие пользователей: 0.44


# === Знакомство: первые персональные рекомендации

In [63]:
ui_data = events[['user_id', 'item_id', 'rating']]

n_users = ui_data['user_id'].nunique()
n_items = ui_data['item_id'].nunique()

all_cells = n_users * n_items
filled_cells = len(ui_data)

empty_cells = all_cells - filled_cells

sparsity = empty_cells / all_cells

print(sparsity)

0.9993451160571009


In [64]:
from surprise import Dataset, Reader
from surprise import SVD

# используем Reader из библиотеки surprise для преобразования событий (events)
# в формат, необходимый surprise
reader = Reader(rating_scale=(1, 5))
surprise_train_set = Dataset.load_from_df(events_train[['user_id', 'item_id', 'rating']], reader)
surprise_train_set = surprise_train_set.build_full_trainset()

# инициализируем модель
svd_model = SVD(n_factors=100, random_state=0)

# обучаем модель
svd_model.fit(surprise_train_set)

In [65]:
surprise_test_set = list(events_test[['user_id', 'item_id', 'rating']].itertuples(index=False))

# получаем рекомендации для тестовой выборки
svd_predictions = svd_model.test(surprise_test_set)

In [66]:
from surprise import accuracy

rmse = accuracy.rmse(svd_predictions)
mae = accuracy.mae(svd_predictions)
                     
print(rmse, mae)

RMSE: 0.8289
MAE:  0.6474
0.8288711689059135 0.647437483750257


In [67]:
from surprise import NormalPredictor

# инициализируем состояние генератора, это необходимо для получения
# одной и той же последовательности случайных чисел, только в учебных целях
np.random.seed(0)

random_model = NormalPredictor()

random_model.fit(surprise_train_set)
random_predictions = random_model.test(surprise_test_set)

In [68]:
rmse_rand = accuracy.rmse(random_predictions)
mae_rand = accuracy.mae(random_predictions)
                     
print(rmse_rand, mae_rand)

RMSE: 1.2628
MAE:  1.0018
1.2628030301013033 1.0017726877569562


In [69]:
events

,user_id,item_id,started_at,read_at,is_read,rating,is_reviewed
0,1229132,22034,2015-07-12,2015-07-17,True,5,False
1,1229132,22318578,2015-06-07,2015-08-09,True,5,True
2,1229132,22551730,2015-06-24,2015-07-11,True,4,True
3,1229132,22816087,2015-09-27,2015-11-04,True,5,True
5,1229132,17910054,2015-03-04,2015-07-28,True,3,False
...,...,...,...,...,...,...,...
12914452,1364473,5297,2017-02-07,2017-02-26,True,5,False
12914453,1364473,4900,2016-12-22,2016-12-29,True,2,False
12914454,1364473,14836,2016-11-29,2017-01-15,True,3,False
12914456,1297020,10210,2012-06-05,2013-01-17,True,5,False


In [76]:
def get_recommendations_svd(user_id, all_items, events, model, include_seen=True, n=5):

    """ возвращает n рекомендаций для user_id """
    
    # получим список идентификаторов всех книг
    all_items = set(events['item_id'].unique())
        
    # учитываем флаг, стоит ли уже прочитанные книги включать в рекомендации
    if include_seen:
        items_to_predict = list(all_items)
    else:
        # получим список книг, которые пользователь уже прочитал ("видел")
        seen_items = set(
            events[
                (events['user_id'] == user_id) &
                (events['is_read'] == True)
            ]['item_id'].unique()
        )
        
        # книги, которые пользователь ещё не читал
        # только их и будем включать в рекомендации
        items_to_predict = list(all_items - seen_items)
    
    # получаем скоры для списка книг, т. е. рекомендации
    predictions = [model.predict(user_id, item_id) for item_id in items_to_predict]
    
    # сортируем рекомендации по убыванию скора и берём только n первых
    recommendations = sorted(predictions, key=lambda x: x.est, reverse=True)[:n]
    
    return pd.DataFrame([(pred.iid, pred.est) for pred in recommendations], columns=["item_id", "score"])

print(get_recommendations_svd(1296647, items, events_test, svd_model))

    item_id     score
0   7864312  4.981188
1  25793186  4.912001
2  12174312  4.898052
3     13208  4.894869
4  33353628  4.891661


In [77]:
# выберем произвольного пользователя из тренировочной выборки ("прошлого")
user_id = events_train['user_id'].sample().iat[0]

print(f"user_id: {user_id}")

print("История (последние события, recent)")
user_history = (
    events_train
    .query("user_id == @user_id")
    .merge(items.set_index("item_id")[["author", "title", "genre_and_votes"]], on="item_id")
)
user_history_to_print = user_history[["author", "title", "started_at", "read_at", "rating", "genre_and_votes"]].tail(10)
display(user_history_to_print)

print("Рекомендации")
user_recommendations = get_recommendations_svd(user_id, items, events_train, svd_model)
user_recommendations = user_recommendations.merge(items[["item_id", "author", "title", "genre_and_votes"]], on="item_id")
display(user_recommendations)

user_id: 1169023
История (последние события, recent)


,author,title,started_at,read_at,rating,genre_and_votes
68,Veronica Roth,"Divergent (Divergent, #1)",2014-06-02,2014-06-04,4,"{'Young Adult': 20260, 'Science Fiction-Dystop..."
69,"Gillian Flynn, В. Русанов",Gone Girl,2014-05-27,2014-05-29,5,"{'Fiction': 11773, 'Mystery': 9965, 'Thriller'..."
70,Kathy Reichs,"Death du Jour (Temperance Brennan, #2)",2014-05-24,2014-05-27,4,"{'Mystery': 1206, 'Mystery-Crime': 579, 'Ficti..."
71,Chelsea Cain,"Heartsick (Archie Sheridan & Gretchen Lowell, #1)",2014-05-22,2014-05-22,5,"{'Mystery': 832, 'Thriller': 653, 'Fiction': 4..."
72,"Jussi Adler-Olsen, Lisa Hartford","The Keeper of Lost Causes (Department Q, #1)",2014-05-30,2014-06-02,3,"{'Mystery': 1225, 'Mystery-Crime': 627, 'Ficti..."
73,Gillian Flynn,Dark Places,2014-05-17,2014-05-22,4,"{'Mystery': 4534, 'Fiction': 4055, 'Thriller':..."
74,Audrey Niffenegger,Her Fearful Symmetry,2014-05-05,2014-05-08,2,"{'Fiction': 1984, 'Fantasy': 674, 'Fantasy-Par..."
75,Kathy Reichs,"Déjà Dead (Temperance Brennan, #1)",2014-05-13,2014-05-17,4,"{'Mystery': 2141, 'Fiction': 904, 'Mystery-Cri..."
76,Carolyn Parkhurst,The Dogs of Babel,2014-05-09,2014-05-10,5,"{'Fiction': 522, 'Mystery': 102, 'Animals': 77..."
77,George R.R. Martin,"A Dance with Dragons (A Song of Ice and Fire, #5)",2014-05-04,2014-05-04,5,"{'Fantasy': 22247, 'Fiction': 4512, 'Fantasy-E..."


Рекомендации


,item_id,score,author,title,genre_and_votes
0,2199,5,Doris Kearns Goodwin,Team of Rivals: The Political Genius of Abraha...,"{'History': 4174, 'Nonfiction': 2127, 'Biograp..."
1,16255632,5,"David Gaider, Ben Gelinas, Mike Laidlaw, Dave ...",Dragon Age: The World of Thedas Volume 1,"{'Fantasy': 134, 'Games-Video Games': 28, 'Art..."
2,2363958,5,João Guimarães Rosa,Grande Sertão: Veredas,"{'Fiction': 85, 'Classics': 69, 'Cultural-Braz..."
3,22552026,5,Jason Reynolds,Long Way Down,"{'Young Adult': 1871, 'Poetry': 1737, 'Contemp..."
4,29237211,5,"Brian K. Vaughan, Fiona Staples","Saga, Vol. 7 (Saga, #7)","{'Sequential Art-Graphic Novels': 2539, 'Seque..."


In [84]:
new_user_id = events["user_id"].max() + 1

new_user_items = [7144, 3, 15881]

# создаём новые события
new_user_events = pd.DataFrame({
    "user_id": [new_user_id] * len(new_user_items),
    "item_id": new_user_items,
    "rating": [5] * len(new_user_items),   # можно поставить свои оценки
    "is_read": [True] * len(new_user_items)
})

# добавляем в events
events_extended = pd.concat(
    [events, new_user_events],
    ignore_index=True
)

In [87]:
recs = get_recommendations_svd(
    user_id=new_user_id,
    all_items=events_extended["item_id"].unique(),
    events=events_extended,
    model=svd_model,
    include_seen=False,
    n=5
)
recs = recs.merge(items[["item_id", "author", "title", "genre_and_votes"]], on="item_id")
display(recs)

,item_id,score,author,title,genre_and_votes
0,24812,5.000000,Bill Watterson,The Complete Calvin and Hobbes,"{'Sequential Art-Comics': 867, 'Humor': 378, '..."
1,11221285,4.914296,Brandon Sanderson,"The Way of Kings, Part 2 (The Stormlight Archi...","{'Fantasy': 641, 'Fiction': 46, 'Fantasy-Epic ..."
2,22037424,4.908423,"J.K. Rowling, Jonny Duddle, Tomislav Tomić",Harry Potter and the Prisoner of Azkaban (Harr...,"{'Fantasy': 49994, 'Young Adult': 15433, 'Fict..."
3,33353628,4.872179,Pénélope Bagieu,"Culottées #2 (Culottées, #2)","{'Sequential Art-Bande DessinÃ©e': 108, 'Femin..."
4,54741,4.872000,Quino,Toda Mafalda,"{'Sequential Art-Comics': 157, 'Humor': 47, 'S..."


# === Базовые подходы: коллаборативная фильтрация

# === Базовые подходы: контентные рекомендации

# === Базовые подходы: валидация

# === Двухстадийный подход: метрики

# === Двухстадийный подход: модель

# === Двухстадийный подход: построение признаков